# ActionShap — Review-8 Remaining Runs

End-to-end notebook for the experiments the KBS reviewers still require. Run on
a machine with the datasets (MovieLens-1M, Amazon Digital Music, and optionally
Gowalla). Each section is independent; run them in any order. Results land in
`results/review8/` and are consumed by the manuscript-integration step.

**Items covered (KBS round-2 mandatory list):**
- **R8-1** LightGCN competitive model — quality gate + full primary audit (Critical #1)
- **R8-2** Gowalla third dataset (round-1 #3)
- **R8-3** Per-user Monte Carlo standard errors at the n=20 cap (round-2 #5)
- **R8-4** Amazon rho=0.25 sensitivity + exhaustive B=3 oracle (round-2 #8)
- **R8-5** Worked user-level examples (round-2 #10)

Everything is pre-wired into `scripts/run_review5_experiments.py` and
`scripts/run_review3_experiments.py`; this notebook just drives them.


In [ ]:
import os, subprocess, sys
from pathlib import Path
CODE = Path(os.getcwd())
if not (CODE / "scripts" / "run_review3_experiments.py").exists():
    CODE = Path("paper-ideas/ActionShap/code")  # opened from repo root
os.chdir(CODE)
PY = sys.executable
OUT = "results/review8"
Path(OUT).mkdir(parents=True, exist_ok=True)
def run(*args):
    print("$", " ".join(args))
    subprocess.run([PY, *args], check=False)
print("cwd:", os.getcwd())


## R8-1. LightGCN — competitive neural model (Critical #1)

LightGCN (`actionshap/lightgcn.py`) uses exact symmetric-normalized graph
propagation with an inference-time history-weighting interface, so masking and
bounded downweighting act without retraining.

**Step 1 — quality gate.** A model may only serve as a robustness cell if it
beats popularity on NDCG@10 and passes the masking gate. Run the full-corpus
variant (30 epochs) on both datasets.

In [ ]:
# LightGCN quality gate, both datasets, full-corpus training (30 epochs)
run("scripts/run_review5_experiments.py", "sasrec-quality", "--model", "lightgcn",
    "--dataset", "movielens", "--train-all", "--epochs", "30", "--out", OUT)
run("scripts/run_review5_experiments.py", "sasrec-quality", "--model", "lightgcn",
    "--dataset", "amazon", "--train-all", "--epochs", "30", "--out", OUT)


**Step 2 — full primary audit.** Only proceed if the gate passes. This runs the
complete ActionShap audit (deletion/bounded alignment, decision quality,
null calibration) with LightGCN as the scorer, 1,000 users, five seeds.

In [ ]:
# Full ActionShap audit under LightGCN (only if the quality gate passes)
run("scripts/run_review3_experiments.py", "--dataset", "movielens",
    "--model", "lightgcn", "--users", "1000", "--out", OUT)
run("scripts/run_review3_experiments.py", "--dataset", "amazon",
    "--model", "lightgcn", "--users", "1000", "--out", OUT)


## R8-6. Tuned LightGCN — attempt to close the quality gate

The full-corpus LightGCN above does **not** beat popularity, because the
protocol requires the user vector to be recomputed at scoring time as a
weighted mean of propagated item embeddings (so bounded downweighting can act
without retraining), which is a weaker recommender than native user-embedding
scoring. This section tries a tuned variant (more layers, larger dim, higher
learning rate, more epochs) to see whether the gate can be passed while
retaining the inference-time weighting interface.

If the gate still fails after tuning, the honest conclusion stands: a
competitive neural recommender that *also* exposes bounded history weighting is
an open engineering requirement, and LightGCN/SASRec are reported as
architecture-applicability cells.


In [ ]:
# Tuned LightGCN quality gate (full-corpus). Sweep a few configurations and
# keep the best that beats popularity + passes the masking gate.
configs = [
    dict(dim=64,  layers=3, lr=0.005, reg=1e-4, epochs=60),
    dict(dim=128, layers=3, lr=0.005, reg=1e-4, epochs=60),
    dict(dim=64,  layers=2, lr=0.01,  reg=1e-5, epochs=80),
]
for ds in ["movielens", "amazon"]:
    for i, c in enumerate(configs):
        tag = f"{ds}-cfg{i}"
        print(f"=== tuned LightGCN {tag} ===")
        run("scripts/run_review5_experiments.py", "sasrec-quality", "--model", "lightgcn",
            "--dataset", ds, "--train-all",
            "--epochs", str(c["epochs"]), "--lgcn-dim", str(c["dim"]),
            "--lgcn-layers", str(c["layers"]), "--lgcn-lr", str(c["lr"]),
            "--lgcn-reg", str(c["reg"]), "--out", OUT)


## R8-2. Gowalla — third dataset, different domain (round-1 #3)

Gowalla is a location check-in dataset (different domain from
entertainment/e-commerce). The prep script downloads the LightGCN-format split
and converts it to the ActionShap temporal CSV (per-user interaction order as
synthetic timestamps; last event = test target, penultimate = validation).

In [ ]:
# Download + convert Gowalla, then run the primary ItemKNN audit
run("scripts/prepare_gowalla.py", "--out", "data/gowalla/interactions.csv")
run("scripts/run_review3_experiments.py", "--dataset", "gowalla",
    "--model", "itemknn", "--users", "1000", "--out", OUT)


## R8-3. Per-user MC standard errors at the n=20 cap (round-2 #5)

Closes the "MC convergence is validated circularly at n=20" gap. For a
subsample of users at the n_max cap, computes:
- per-player Monte Carlo SEs of the Shapley values (from the sampled marginal
  contributions), with relative SE vs the largest |phi|;
- an **independent** larger-budget reference (different seed stream,
  M_pair=1000) and its rank correlation with the primary-budget estimate —
  a non-circular convergence check at n=20.

In [ ]:
run("scripts/run_review5_experiments.py", "mcse-n20", "--dataset", "movielens",
    "--users", "1000", "--subsample", "50", "--permutations", "250",
    "--ref-permutations", "1000", "--out", OUT)
run("scripts/run_review5_experiments.py", "mcse-n20", "--dataset", "amazon",
    "--users", "1000", "--subsample", "50", "--permutations", "250",
    "--ref-permutations", "1000", "--out", OUT)


## R8-4. Amazon rho=0.25 + exhaustive B=3 (round-2 #8)

**rho=0.25 sensitivity** on Amazon (the MovieLens cell already exists).
**Exhaustive B=3 oracle** (1,351 actions/user — computationally trivial)
replacing the greedy lower bound, with additive-selection regret at B=2 and B=3.

In [ ]:
# Amazon rho=0.25 sensitivity cell
run("scripts/run_review3_experiments.py", "--dataset", "amazon",
    "--rho", "0.25", "--users", "250", "--out", OUT)


In [ ]:
# Exhaustive B=3 oracle + regret (both datasets)
run("scripts/run_review5_experiments.py", "exhaustive-b3", "--dataset", "movielens",
    "--users", "1000", "--subsample", "200", "--permutations", "250", "--out", OUT)
run("scripts/run_review5_experiments.py", "exhaustive-b3", "--dataset", "amazon",
    "--users", "1000", "--subsample", "200", "--permutations", "250", "--out", OUT)


## R8-5. Worked user-level examples (round-2 #10)

Per-user case studies showing the protocol's knowledge-level output: profile
players (item ids), top attributions with deletion vs bounded effects, the
method's selected action vs the exact budget-2 oracle, and realized effects.

In [ ]:
run("scripts/run_review5_experiments.py", "worked-example", "--dataset", "movielens",
    "--examples", "3", "--permutations", "250", "--top-k", "5", "--out", OUT)
run("scripts/run_review5_experiments.py", "worked-example", "--dataset", "amazon",
    "--examples", "3", "--permutations", "250", "--top-k", "5", "--out", OUT)


## Done

Push the contents of `results/review8/` and ping for manuscript integration:
- `lightgcn_quality_*.json`, `lightgcn` audit JSONs (R8-1)
- `gowalla` audit JSON (R8-2)
- `mcse_n20_*.json` (R8-3)
- Amazon `rho0.25` audit + `exhaustive_b3_*.json` (R8-4)
- `worked_examples_*.json` (R8-5)
